## Without web search

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import os

model = init_chat_model(
    model="agnes-2.0-flash",
    model_provider="openai",
    base_url=os.getenv("AGNES_BASE_URL"),
    api_key=os.getenv("AGNES_API_KEY"),
)

agent = create_agent(
    model=model
)

In [4]:
from langchain.messages import HumanMessage

question = HumanMessage(content="请问深圳市市长是谁")

response = agent.invoke(
    {"messages": [question]}
)

In [5]:
print(response['messages'][-1].content)

截至 2024 年，深圳市市长是**覃伟中**。

他于 2021 年 5 月被任命为深圳市代市长，并在同年 6 月正式当选为深圳市市长。请注意，政府职务可能会随时间调整，如需获取最新信息，建议查阅深圳市人民政府官方网站或权威新闻媒体。


## Add web search tool

In [9]:
from langchain.tools import tool
from typing import Dict, Any
from ddgs import DDGS

@tool
def web_search(query: str) -> list[Dict[str, str]]:
    """Search the web for information using DuckDuckGo (free, no API key needed)"""
    results = []
    with DDGS() as search:
        for r in search.text(query, max_results=5):
            results.append({
                "title": r["title"],
                "url": r["href"],
                "content": r["body"],
            })
    return results

web_search.invoke("深圳市市长是谁")

[{'title': '深圳市市长列表 - 维基百科，自由的百科全书',
  'url': 'https://zh.wikipedia.org/zh-hans/深圳市市長列表',
  'content': '深圳市人民政府当前的行政首长是深圳市人民政府市长，现任市长为 覃伟中，于2021年4月24日开始代理 [5]，5月19日正式出任 [6]。 深圳特区成立后首三任市长均为广东人，包括来自 汕头 的 吴南生 、 开平 的 梁湘，以及 茂名 （电白 [7]）的 李灏。'},
 {'title': '覃伟中 - 维基百科，自由的百科全书',
  'url': 'https://zh.wikipedia.org/wiki/覃伟中',
  'content': '覃伟中 （1971年7月3日—），男，汉族， 广西 玉林 人，生于 吉林省 吉林市， 中华人民共和国 政治人物。 清华大学 在职工学博士学位，高级工程师。现任 中共深圳市委 副书记、 深圳市人民政府 市长，是 第二十届中共中央候补委员。'},
 {'title': '覃伟中_百度百科',
  'url': 'https://baike.baidu.com/item/覃伟中/20597172',
  'content': '覃伟中，男，汉族，1971年7月生，广西玉林人，1996年7月参加工作，2001年6月加入中国共产党，清华大学化学工程系化学工程与技术专业毕业，研究生学历，工学博士学位，高级工程师。现任第二十届中央候补委员，广东省深圳市委副书记，市政府市长、党组书记。'},
 {'title': '覃伟中-深圳政府在线_深圳市人民政府门户网站',
  'url': 'https://www.sz.gov.cn/cn/xxgk/zfxxgj/sldzc/sz/qwz/index.html',
  'content': '市领导之窗 您现在的位置 首页 > 政务公开 > 政府信息公开 > 领导成员 > 市长 > 覃伟中 覃伟中 市长 覃伟中，男，汉族，1971年7月生，研究生学历、工学博士，中共党员。 现任二十届中央候补委员，深圳市委副书记，市政府市长、党组书记。'},
 {'title': '赵勇（广东省深圳市委常委、市纪委书记、市监委主任）_百度百科',
  'url': 'https://baike.baidu

In [10]:
agent = create_agent(
    model=model,
    tools=[web_search]
)

question = HumanMessage(content="请问深圳市市长是谁")

response = agent.invoke(
    {"messages": [question]}
)

In [11]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='请问深圳市市长是谁', additional_kwargs={}, response_metadata={}, id='1b6f4ed6-fc6f-4bfb-aa59-503609c744cf'),
 AIMessage(content='截至 2024 年，深圳市市长是**覃伟中**。\n\n他于 2021 年 6 月被任命为深圳市市长，并在此后继续担任该职务。如果需要最新信息，建议查阅深圳市人民政府官方网站，因为政府人事变动可能会发生。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 478, 'total_tokens': 541, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'agnes-2.0-flash', 'system_fingerprint': 'vllm-0.21.0-tp2-52bef988', 'id': 'chatcmpl-a140f1be1fd7b9e8', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019ebc3d-b2a6-7862-943a-f93772cd9358-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 478, 'output_tokens': 63, 'total_tokens': 541, 'input_token_details': {}, 'output_token_details': {}})]


trace: https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r